# Quantum Autoencoder — Detailed Notes (Session 22)
**Course:** CS490/5590 — Quantum Computing Applications in Data Science, AI, & Deep Learning  
**Instructor:** Luke Miller

> **Purpose.** These notes turn the slide bullets into a stand-alone reference on **quantum autoencoders (QAEs)**. You’ll learn how to design encoder/decoder PQCs, choose losses (trash-state vs. fidelity), train with modern Qiskit **primitives** (Sampler/Estimator), and evaluate compression/anomaly-detection performance on NISQ-era devices (or simulators).

---

## Session roadmap
1. Recap: QGANs → why compression next  
2. What is a quantum autoencoder? (concept & goals)  
3. Architectures: encoder/latent/decoder + “trash” qubits  
4. Losses: **trash-state** probability, **fidelity** via swap test, **observable** losses  
5. Training strategies: gradient-free (SPSA) vs. gradient-based (parameter-shift / EstimatorQNN)  
6. Qiskit implementations (primitives, modern APIs)  
7. Applications: compression, denoising, anomaly detection  
8. Evaluation protocol & metrics  
9. Practical tips & pitfalls  
10. Mini-exercises (with brief answers)

---

## 0) Recap → From QGANs to QAEs
- **QGANs** *synthesize* data; **QAEs** *compress & reconstruct* data.  
- In quantum pipelines, QAEs help **reduce qubit count**, **suppress noise**, and **extract features** for downstream QML (QNN/QSVM/QCNN).

---

## 1) What is a Quantum Autoencoder?
- **Goal.** Learn a unitary (or near-unitary) **encoder** $U_E(\theta)$ that maps an $n$-qubit input state $|\psi\rangle$ to a **latent** $m$-qubit state $|\phi\rangle$ plus **trash** qubits driven to $|0\cdots0\rangle$. A **decoder** $U_D(\phi)$ reconstructs $|\hat\psi\rangle\approx|\psi\rangle$.
- **Compression.** $m<n$ ⇒ fewer qubits carry the informative content.
- **Two common objectives**
  1) **Trash-state objective**: after $U_E$, maximize $P(\text{trash}=\mathbf{0})$.  
  2) **Reconstruction fidelity**: after $U_D\circ U_E$, maximize $F=|\langle\psi|\hat\psi\rangle|^2$.

> NISQ-friendly trick: train **only the encoder** with the **trash-state objective** (no decoder), then (optionally) learn $U_D$ or set $U_D\approx U_E^\dagger$.

---

## 2) Architecture: Encoder / Latent / Decoder
- **Encoder $U_E(\theta)$**: shallow **TwoLocal** (e.g., `ry`/`rz` + `cx`), 1–2 reps, entanglement pattern matching hardware topology.  
- **Latent $m$**: select a subset of qubits to retain; remaining $t=n-m$ are “trash.”  
- **Decoder $U_D(\phi)$**: often **the inverse** $U_E^\dagger$ or a trainable PQC mirroring the encoder.

**Data preparation.**
- **Quantum data**: given or prepared states (e.g., toy molecules, Bell pairs).  
- **Classical data**: encode via **angle** (feature) maps or **amplitude** encoding (deeper).

---

## 3) Losses for training

### (A) Trash-state probability (local objective, NISQ-friendly)
After encoding, measure the trash register. Define  
$$
\mathcal{L}_{\text{trash}}(\theta) \;=\; 1 - P_\theta(\text{trash}=\mathbf{0}).
$$
- Efficient: only measure a few qubits; no decoder needed.
- Works with mixed/noisy inputs; encourages **concentration of information** into latent qubits.

### (B) Reconstruction fidelity (global objective)
Apply $U_D(\phi)\circ U_E(\theta)$ and compute  
$$
\mathcal{L}_{\text{fid}}(\theta,\phi) \;=\; 1 - |\langle\psi|\hat\psi\rangle|^2.
$$
Estimate fidelity with a **swap test** between $|\psi\rangle$ and $|\hat\psi\rangle$ or by measuring a suitable **projector**.

### (C) Observable losses (Pauli expansions)
For known target states $\rho$, compare **expectations**:  
$$
\mathcal{L}_{\text{obs}} = \sum_j w_j\Big(\langle O_j\rangle_{\hat\psi} - \langle O_j\rangle_{\psi}\Big)^2.
$$
This reduces circuit overhead if a small operator set $\{O_j\}$ characterizes the data manifold.

---

## 4) Training strategies

### Gradient-free (robust on hardware)
- **SPSA** or Nelder–Mead optimize $\mathcal{L}(\theta)$ using only function evaluations.  
- Pros: resilient to shot/readout noise. Cons: more iterations.

### Gradient-based (primitives / parameter-shift)
- Use **Estimator** with parameter-shift to get $\partial_\theta\langle O\rangle$.  
- Pros: faster convergence for clean simulators. Cons: sensitive to noise; careful shot budgeting required.

> On NISQ, start with **trash-state loss + SPSA**. Add decoder/fidelity later if resources allow.

---

## 5) Qiskit implementations (modern primitives)

> We show two recipes: (1) **Trash-state training** (no decoder), and (2) **Fidelity via swap test** (for evaluation). Code uses **Aer** and **primitives**.

### 5.1 Setup & small encoder (4→2 compression)
```python
# pip install qiskit qiskit-aer
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import TwoLocal
from qiskit_aer.primitives import Sampler, Estimator
from qiskit.quantum_info import SparsePauliOp

rng = np.random.default_rng(7)
sampler = Sampler()
estimator = Estimator()

n, m = 4, 2          # 4 input qubits -> 2 latent, 2 trash
latent = [0,1]       # keep qubits 0,1 as latent
trash  = [2,3]       # drive to |00>

# Example encoder ansatz (shallow)
encoder = TwoLocal(n_qubits=n,
                   rotation_blocks=['ry','rz'],
                   entanglement_blocks='cx',
                   entanglement='linear',
                   reps=1, insert_barriers=False)
theta = np.zeros(encoder.num_parameters)  # identity init = plateau mitigation
```

### 5.2 Prepare a *family* of training states
```python
def prep_input(idx:int)->QuantumCircuit:
    """
    Toy data manifold: small set of 4-qubit states:
    - product states and light entanglement patterns
    """
    qc = QuantumCircuit(n)
    if idx % 3 == 0:
        # simple product state |+0+1>
        qc.h(0); qc.h(2); qc.x(3)
    elif idx % 3 == 1:
        # one Bell pair on (1,2)
        qc.h(1); qc.cx(1,2)
    else:
        # weakly entangled (0,1) with RYY, then X on 3
        qc.ryy(0.4, 0, 1); qc.x(3)
    return qc
```

### 5.3 **Trash-state loss** via Estimator (projector onto |00〉 on trash)
Probability that **both trash qubits are |0〉** equals the expectation of  
$$
P_{00}=\frac{1}{4}(I+Z_2)\otimes(I+Z_3)
$$
(extended by $I$ on latent qubits). Build it as a Pauli sum:
```python
def projector_trash00(n, trash):
    # Build P = ⊗_{q in trash} (I+Z_q)/2  and identity on others
    # For t=2, expands to 4 terms: II + IZ + ZI + ZZ (on trash positions)
    # We'll tensor identities elsewhere.
    # Represent as SparsePauliOp sum with weights 1/4 each.
    terms = []
    coeffs = []
    # Enumerate {0,1}^t to select which Zs appear
    from itertools import product
    for bits in product([0,1], repeat=len(trash)):
        label = ['I']*n
        for b, q in zip(bits, trash):
            label[q] = 'Z' if b==1 else 'I'
        terms.append(''.join(label[::-1]))   # Qiskit uses little-endian pauli ordering
        coeffs.append(1/4)
    return SparsePauliOp.from_list(list(zip(terms, coeffs)))

P_trash00 = projector_trash00(n, trash)
```

Evaluate **trash-state probability** for a batch of inputs:
```python
def trash_zero_prob(theta, batch_ids):
    # Compose (input prep) -> encoder(theta)
    circuits = []
    for idx in batch_ids:
        qc = QuantumCircuit(n)
        qc.compose(prep_input(idx), inplace=True)
        qc.compose(encoder.bind_parameters(theta), inplace=True)
        circuits.append(qc)
    # Expectation of projector
    res = estimator.run(circuits, [P_trash00]*len(circuits)).result()
    vals = np.array(res.values, dtype=float)
    return float(vals.mean())   # average over batch
```

### 5.4 SPSA training loop (gradient-free)
```python
def spsa_train(theta, steps=200, a0=0.15, c0=0.1, batch_size=4, seed=1):
    rng = np.random.default_rng(seed)
    loss_history = []
    for k in range(1, steps+1):
        a = a0 / (k**0.602)       # standard SPSA schedules
        c = c0 / (k**0.101)

        # Mini-batch of training states
        batch_ids = rng.integers(0, 24, size=batch_size)

        # Random Rademacher perturbation
        delta = rng.choice([-1.0, 1.0], size=theta.size)

        p_plus  = trash_zero_prob(theta + c*delta, batch_ids)
        p_minus = trash_zero_prob(theta - c*delta, batch_ids)

        # Loss = 1 - P(trash==00)
        L_plus, L_minus = 1 - p_plus, 1 - p_minus
        ghat = (L_plus - L_minus) / (2*c*delta)  # elementwise

        theta = theta - a * ghat                  # update
        # Monitor
        with np.errstate(all='ignore'):
            p_now = trash_zero_prob(theta, batch_ids)
        loss_history.append(1 - p_now)

        if k % 20 == 0:
            print(f"[SPSA] step {k:3d} | loss≈{1-p_now:.4f} | P00≈{p_now:.4f}")
    return theta, loss_history

theta_trained, losses = spsa_train(theta.copy(), steps=120)
```

### 5.5 (Optional) **Fidelity** evaluation via **swap test** (encoder+decoder)
Use the inverse encoder as decoder $U_D\approx U_E^\dagger$ and a swap test vs. the original input:
```python
def swap_test_fidelity(theta, idx):
    # Build |psi> and |hat{psi}> = U_D U_E |psi>, with U_D = U_E^\dagger
    qc_enc = QuantumCircuit(n)
    qc_enc.compose(prep_input(idx), inplace=True)
    qc_enc.compose(encoder.bind_parameters(theta), inplace=True)

    # Decoder as inverse (acts on same n qubits)
    qc_dec = QuantumCircuit(n)
    qc_dec.compose(encoder.bind_parameters(theta).inverse(), inplace=True)

    # Compose swap test on ancilla + two n-qubit regs:
    anc = 1; reg = n
    from qiskit import QuantumCircuit as QC
    st = QC(1 + 2*n)
    # prepare |psi> on first reg, |hat{psi}> on second
    st.compose(qc_enc, qubits=range(1, 1+n), inplace=True)
    tmp = QC(n); tmp.compose(prep_input(idx), inplace=True); tmp.compose(encoder.bind_parameters(theta), inplace=True); tmp.compose(qc_dec, inplace=True)
    st.compose(tmp, qubits=range(1+n, 1+2*n), inplace=True)
    # swap test
    st.h(0)
    for q in range(n):
        st.cswap(0, 1+q, 1+n+q)
    st.h(0)

    # Probability ancilla=0: p0 = (1+F)/2  =>  Fidelity F = 2*p0 - 1
    res = sampler.run([st], shots=4096).result().quasi_dists[0]
    p0 = res.get(0, 0.0)
    F = max(0.0, min(1.0, 2*p0 - 1))
    return F
```

---

## 6) Applications

### Dimensionality reduction & compression
- Keep $m$ qubits; **discard trash**. Downstream models (QNN/QSVM/QCNN) train on latent only.  
- Report **compression ratio** $n/m$ and task accuracy delta.

### Denoising
- Train on **clean** states; at inference, encode **noisy** states, then **decode** (or directly use latent). Improvement = higher fidelity / lower task loss.

### Anomaly detection
- High **reconstruction loss** (or low trash-state probability) flags out-of-distribution inputs.

---

## 7) Evaluation protocol & metrics
- **Training curves**: trash-prob ↑, loss ↓.  
- **Reconstruction fidelity**: mean/median $F$ on validation set.  
- **Latent task**: accuracy of downstream classifier using latent qubits vs. baseline.  
- **Robustness**: sensitivity to noise (add depolarizing/readout noise in Aer).  
- **Resource report**: qubit count, depth, shots, transpiler level.

---

## 8) Practical tips & pitfalls
- **Start near identity** (zeros for angles) to avoid barren plateaus.  
- **Shallow first**: reps=1; add depth only if loss plateaus.  
- **Batch small** (2–8 states) for stable SPSA noise averaging.  
- **Readout mitigation** helps when measuring trash register.  
- **Feature scaling** for classical encodings (e.g., map to $[-\pi,\pi]$).  
- **Cache transpiles** and **bind parameters** to avoid recompilation.

---

## 9) Mini-exercises (answers below)
1. **Projector derivation.** Show that for trash set $T$, $P(\mathbf{0}_T)=\bigotimes_{q\in T}(I+Z_q)/2$.  
2. **Latent-only training.** Describe how you’d add a small **latent head** (Pauli Z expectations) to regularize the encoder to preserve class info.  
3. **Noise stress test.** Add 1% depolarizing noise to 2-qubit entanglers; predict impact on trash-prob curves.  
4. **Decoder ablation.** Compare $U_D=U_E^\dagger$ vs. a trainable decoder: when might the latter help?  
5. **Anomaly threshold.** Propose a principled threshold on $\mathcal{L}_{\text{trash}}$ for flagging anomalies without labels.

---

## 10) Summary
- QAEs **compress** quantum/classical data (after encoding) by pushing **trash** qubits to $|0\cdots0\rangle$ and preserving information in a **latent** register.  
- On NISQ, the **trash-state objective + SPSA** is the simplest effective training strategy.  
- Evaluate with **fidelity**, **latent-task accuracy**, and **robustness** under noise.  
- QAEs pair naturally with QGANs (evaluate generations) and as a preprocessing step for QNN/QSVM.

---

## Looking ahead
- **Next Session:** Final Review & Project Clinic — integrate QAEs/QGANs/QKernels into a coherent QML pipeline.  
- **Homework 5 (Autoencoder):**  
  1) Train a 4→2 QAE with trash-state loss on a small state family; report $P_{\text{trash}=0}$ curves.  
  2) Evaluate reconstruction fidelity via swap test on a held-out set.  
  3) (Bonus) Add depolarizing noise and test denoising performance.

---

### Appendix — Mini-exercise solutions (sketch)
1. Each qubit’s projector onto $|0\rangle$ is $(I+Z)/2$. Tensoring across the trash set and identity elsewhere yields $P(\mathbf{0}_T)$.  
2. Add a latent readout head measuring $\langle Z_i\rangle$ on latent qubits and add an auxiliary loss aligning them with class labels (semi-supervised) or maximizing variance (unsupervised).  
3. Depolarizing noise reduces constructive interference ⇒ **lower** $P_{\text{trash}=0}$, slower convergence; mitigation or shorter depth helps.  
4. If the data manifold is not exactly unitary-encodable by the chosen ansatz, a **trainable decoder** can compensate for encoder bias, improving fidelity.  
5. Fit a Gaussian or EVT model to training losses; set threshold at, e.g., mean + 3σ (or pick percentile like 99th) for controlled false-positive rate.
